# CALM-VAD — pose extraction + evaluation (Colab)

**What this notebook does**
1. installs deps and pulls the repo
2. gets a benchmark's videos (Drive mount, direct URL, or manual upload)
3. runs `calm.extract_poses` on the **GPU** → cached skeleton JSON
4. runs `calm.harness` (CPU-light) → the 5-axis report
5. zips `data/pose/` + `results/` for download

The only GPU step is #3. Everything after can also be re-run on your laptop.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

## 1 · Environment

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics scikit-learn scipy pyyaml opencv-python-headless
import torch; print('CUDA available:', torch.cuda.is_available())

## 2 · Get the code

**Option A — GitHub** (recommended: push the `sentrix/` repo, then set the URL).

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/sentrix.git'  — leave '' to upload a zip instead
import os, sys
if REPO_URL:
    !git clone --depth 1 $REPO_URL /content/sentrix
else:
    from google.colab import files
    print('Upload a zip of the sentrix/ project (must contain the calm/ folder):')
    up = files.upload()
    zname = next(iter(up))
    !mkdir -p /content/sentrix && unzip -q "$zname" -d /content/sentrix
    # if the zip has a top-level folder, flatten it
    subs = [d for d in os.listdir('/content/sentrix') if os.path.isdir(f'/content/sentrix/{d}')]
    if 'calm' not in subs and len(subs) == 1:
        inner = f'/content/sentrix/{subs[0]}'
        !cp -r "$inner"/* /content/sentrix/
os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
assert os.path.isdir('calm'), 'calm/ not found — check the repo/zip layout'
print('cwd:', os.getcwd()); print(sorted(os.listdir('calm')))

In [ ]:
# sanity: the maths works before we touch any data (~2 s)
!python -m calm.selftest

## 3 · Get a benchmark's videos

Pick **one** path. Put the video files under `/content/data/videos/` and any
per-clip ground truth under `/content/data/gt/` (filename stems must match).

GT formats accepted: `*.npy` frame mask (1=anomaly), `*.txt` (one `start end`
per line, frame indices), or `*.json` `{"gt": [[s,e],...]}`.

Common sources (verify — dataset links move):
* **ShanghaiTech**: https://svip-lab.github.io/dataset/campus_dataset.html
* **CUHK Avenue**: http://www.cse.cuhk.edu.hk/leojia/projects/detectabnormal/dataset.html
* **UBnormal**: https://github.com/lilygeorgescu/UBnormal
* **NWPU Campus**: https://campusvad.github.io/
* **CHAD**: https://github.com/TeCSAR-UNCC/CHAD

In [ ]:
!mkdir -p /content/data/videos /content/data/gt /content/sentrix/data/pose /content/sentrix/results

SOURCE = 'drive'   # 'drive' | 'url' | 'upload'

if SOURCE == 'drive':
    from google.colab import drive; drive.mount('/content/drive')
    # EDIT these to point at your copies in Drive:
    VID_DIR = '/content/drive/MyDrive/datasets/shanghaitech/testing/videos'
    GT_DIR  = '/content/drive/MyDrive/datasets/shanghaitech/testing/test_frame_mask'
    !cp -v "$VID_DIR"/* /content/data/videos/ 2>/dev/null | tail -3
    !cp -v "$GT_DIR"/*  /content/data/gt/     2>/dev/null | tail -3

elif SOURCE == 'url':
    URL = ''  # a direct link to a .zip / .tar of videos
    !wget -q --show-progress -O /content/ds.zip "$URL" && unzip -q /content/ds.zip -d /content/data/videos

elif SOURCE == 'upload':
    from google.colab import files
    print('Upload video files (repeat the cell for GT):')
    for n in files.upload(): os.rename(n, f'/content/data/videos/{n}')

import glob
print('videos:', len(glob.glob('/content/data/videos/*')), ' gt:', len(glob.glob('/content/data/gt/*')))

## 4 · Extract poses (GPU) — the one step that needs it

Run once per split. For the paper you want a `calib` split (some normal + some
anomaly clips) and a `test` split — easiest is to extract everything as `test`
and let the harness carve the calibration set, or split the video folders
yourself and run this cell twice with different `--split`.

In [ ]:
TAG = 'shanghaitech'
!python -m calm.extract_poses \
  --videos /content/data/videos \
  --gt     /content/data/gt \
  --out    /content/sentrix/data/pose/$TAG.json \
  --split  test \
  --weights yolo11n-pose.pt --imgsz 640 --device 0 --stride 1
# faster/cheaper: --stride 2 --imgsz 512

## 5 · Evaluate (CPU-light) — the 5-axis report

In [ ]:
!python -m calm.harness --generic /content/sentrix/data/pose/$TAG.json --tag $TAG
print('\n--- JSON ---')
import json; print(json.dumps(json.load(open(f'/content/sentrix/results/calm_report_{TAG}.json')), indent=2)[:4000])

## 6 · Cross-dataset run (optional)

Extract a second benchmark, then merge the two pose files with one set to
`split=calib` and the other `split=test` and run the harness once.

In [ ]:
import json
def merge(fit_json, test_json, out_json):
    a = json.load(open(fit_json)); b = json.load(open(test_json))
    for c in a['clips']: c['split'] = 'calib'
    for c in b['clips']: c['split'] = 'test'
    json.dump({'fps': a['fps'], 'clips': a['clips'] + b['clips']}, open(out_json, 'w'))
    print('wrote', out_json)
# merge('data/pose/shanghaitech.json','data/pose/nwpu.json','data/pose/sht_to_nwpu.json')
# !python -m calm.harness --generic data/pose/sht_to_nwpu.json --tag sht_to_nwpu

## 7 · Download everything

In [ ]:
!cd /content/sentrix && zip -qr /content/calm_out.zip data/pose results
from google.colab import files; files.download('/content/calm_out.zip')
# also copy to Drive so a disconnect doesn't lose the poses:
!cp /content/calm_out.zip /content/drive/MyDrive/ 2>/dev/null && echo 'copied to Drive'